<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/Decouple_SEAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 13.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.0 MB/s eta 0:00:00
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=677287 sha256=39d3417da3e40149531f39eb529a74157d42fe816b7d84fac7c682ea4f96d050
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
  Created wheel for torch-sparse: filename=torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl size=1261482 sha256=b30a3d7585c67cd5cb181ee7c0cb7f05da4db3e2c

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from torch_geometric.utils import negative_sampling, k_hop_subgraph
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import SAGEConv, global_sort_pool
from sklearn.metrics import roc_auc_score, average_precision_score


import networkx as nx

In [3]:
import torch
import numpy as np
import pandas as pd

content_path = "/content/sample_data/cora.content"
# Load content file
content = pd.read_csv(content_path, sep="\t", header=None)

paper_ids = content[0].values
features = content.iloc[:, 1:-1].values
labels_raw = content.iloc[:, -1].values

# Map paper IDs to indices
id_map = {id_: i for i, id_ in enumerate(paper_ids)}

# Encode labels
label_map = {label: i for i, label in enumerate(set(labels_raw))}
labels = np.array([label_map[l] for l in labels_raw])

x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)

In [4]:
cites_path = "/content/sample_data/cora.cites"

cites = pd.read_csv(cites_path, sep="\t", header=None)

edges = []

for _, row in cites.iterrows():
    src, dst = row[0], row[1]
    if src in id_map and dst in id_map:
        edges.append([id_map[src], id_map[dst]])

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

# Make undirected
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

print("Nodes:", x.shape[0])
print("Edges:", edge_index.shape[1])



Nodes: 2708
Edges: 10858


In [5]:
from torch_geometric.data import Data

data = Data(x=x, edge_index=edge_index, y=y)

In [13]:
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True,
    split_labels=False   # IMPORTANT
)

train_data, val_data, test_data = transform(data)

In [21]:
def drnl_node_labeling(edge_index, src, dst, num_nodes):

    import networkx as nx

    G = nx.Graph()

    # Add ALL nodes first (important)
    G.add_nodes_from(range(num_nodes))

    edges = edge_index.t().tolist()
    G.add_edges_from(edges)

    dist_src = nx.single_source_shortest_path_length(G, src)
    dist_dst = nx.single_source_shortest_path_length(G, dst)

    z = torch.zeros(num_nodes)

    for i in range(num_nodes):

        d1 = dist_src.get(i, 1e6)
        d2 = dist_dst.get(i, 1e6)

        if d1 == 1e6 or d2 == 1e6:
            z[i] = 0
        else:
            z[i] = 1 + min(d1, d2) + (d1 + d2)//2 * ((d1 + d2)//2 + (d1 + d2)%2 - 1)

    return z.long()

In [22]:
def extract_subgraph(src, dst, graph_data, label, num_hops=2):

    subset, edge_index_sub, mapping, _ = k_hop_subgraph(
        [src, dst],
        num_hops,
        graph_data.edge_index,
        relabel_nodes=True
    )

    x = graph_data.x[subset]

    src_new, dst_new = mapping

    z = drnl_node_labeling(edge_index_sub, src_new.item(), dst_new.item(), subset.size(0))

    return Data(
        x=x,
        edge_index=edge_index_sub,
        z=z,
        y=torch.tensor([label])
    )

In [23]:
def build_dataset(graph_data):

    dataset = []

    edge_index = graph_data.edge_label_index
    labels = graph_data.edge_label

    for i in range(edge_index.shape[1]):

        src = edge_index[0, i].item()
        dst = edge_index[1, i].item()
        label = labels[i].item()

        dataset.append(
            extract_subgraph(src, dst, graph_data, label)
        )

    return dataset

In [24]:
train_dataset = build_dataset(train_data)
val_dataset = build_dataset(val_data)
test_dataset = build_dataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

/tmp/ipykernel_1908/844833640.py:5: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
/tmp/ipykernel_1908/844833640.py:6: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader = DataLoader(val_dataset, batch_size=32)
/tmp/ipykernel_1908/844833640.py:7: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(test_dataset, batch_size=32)


In [25]:
class DSEAL(nn.Module):

    def __init__(self, in_channels, hidden=64, k=30):

        super().__init__()

        self.z_embedding = nn.Embedding(100, hidden)

        self.conv1 = SAGEConv(in_channels + hidden, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, hidden)

        self.k = k

        self.lin1 = nn.Linear(hidden * k, 128)
        self.lin2 = nn.Linear(128, 1)

    def forward(self, data):

        x, edge_index, z, batch = data.x, data.edge_index, data.z, data.batch

        z_emb = self.z_embedding(z)

        x = torch.cat([x, z_emb], dim=1)

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))

        x = global_sort_pool(x, batch, self.k)

        x = F.relu(self.lin1(x))
        x = self.lin2(x)

        return torch.sigmoid(x).view(-1)

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DSEAL(train_data.x.size(1)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [27]:
def train():

    model.train()
    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)

        pred = model(batch)

        loss = F.binary_cross_entropy(pred, batch.y.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [28]:
@torch.no_grad()
def evaluate(loader):

    model.eval()

    preds = []
    labels = []

    for batch in loader:

        batch = batch.to(device)

        pred = model(batch)

        preds.append(pred.cpu())
        labels.append(batch.y.cpu())

    preds = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()

    auc = roc_auc_score(labels, preds)
    ap = average_precision_score(labels, preds)

    return auc, ap

In [30]:
for epoch in range(1, 21):

    loss = train()

    val_auc, val_ap = evaluate(val_loader)

    print(f"Epoch {epoch}")
    print("Loss:", loss)
    print("Val AUC:", val_auc)
    print("Val AP:", val_ap)

/tmp/ipykernel_1908/2837286930.py:30: UserWarning: 'nn.glob.global_sort_pool' is deprecated, use 'nn.aggr.SortAggr' instead
  x = global_sort_pool(x, batch, self.k)


Epoch 1
Loss: 3.874067310986363e-08
Val AUC: 0.7759528056535178
Val AP: 0.7821592018404286


/tmp/ipykernel_1908/2837286930.py:30: UserWarning: 'nn.glob.global_sort_pool' is deprecated, use 'nn.aggr.SortAggr' instead
  x = global_sort_pool(x, batch, self.k)


Epoch 2
Loss: 1.99012623141311e-08
Val AUC: 0.7768276575754688
Val AP: 0.781541440300823


/tmp/ipykernel_1908/2837286930.py:30: UserWarning: 'nn.glob.global_sort_pool' is deprecated, use 'nn.aggr.SortAggr' instead
  x = global_sort_pool(x, batch, self.k)


...

Epoch 18
Loss: 6.202218653503938e-10
Val AUC: 0.7661336991598698
Val AP: 0.7690218228261997


/tmp/ipykernel_1908/2837286930.py:30: UserWarning: 'nn.glob.global_sort_pool' is deprecated, use 'nn.aggr.SortAggr' instead
  x = global_sort_pool(x, batch, self.k)


Epoch 19
Loss: 5.388380908721073e-10
Val AUC: 0.7663294345120573
Val AP: 0.7687472640952129


/tmp/ipykernel_1908/2837286930.py:30: UserWarning: 'nn.glob.global_sort_pool' is deprecated, use 'nn.aggr.SortAggr' instead
  x = global_sort_pool(x, batch, self.k)


Epoch 20
Loss: 4.884693261804857e-10
Val AUC: 0.7650341770945385
Val AP: 0.7676861822994225


In [31]:
test_auc, test_ap = evaluate(test_loader)

print("Test AUC:", test_auc)
print("Test AP:", test_ap)

/tmp/ipykernel_1908/2837286930.py:30: UserWarning: 'nn.glob.global_sort_pool' is deprecated, use 'nn.aggr.SortAggr' instead
  x = global_sort_pool(x, batch, self.k)


Test AUC: 0.7787084190030092
Test AP: 0.7874181412838499
